In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/meruvakodandasuraj/youtube-trending-videos-20202026/category_summary.csv
/kaggle/input/datasets/meruvakodandasuraj/youtube-trending-videos-20202026/yearly_trends.csv
/kaggle/input/datasets/meruvakodandasuraj/youtube-trending-videos-20202026/country_summary.csv
/kaggle/input/datasets/meruvakodandasuraj/youtube-trending-videos-20202026/trending_videos.csv


In [8]:
import pandas as pd
import plotly.express as px

# ... (Previous loading code remains the same) ...

try:
    # 1. FIXED: Global Market Reach (Using 'trending_country' instead of 'country')
    fig_country = px.pie(df_country, 
                        values='avg_views', 
                        names='trending_country', # Fixed column name
                        title="Global Market Share: Average Views by Country",
                        hole=0.4, 
                        template="plotly_dark")
    fig_country.show()

    # 2. FIXED: Category Efficiency (Using 'avg_engagement' directly)
    # The dataset already calculated engagement for you!
    fig_cat = px.bar(df_category.sort_values('avg_engagement', ascending=False), 
                    x='category', # If this fails, try 'top_category'
                    y='avg_engagement',
                    title="Average Engagement Rate by Category",
                    color='avg_engagement', 
                    color_continuous_scale='Reds',
                    template="plotly_dark")
    fig_cat.show()

    # 3. Yearly Growth
    # Check if 'year' is correct or if it's 'publish_year'
    fig_year = px.line(df_yearly, x='year', y='avg_views',
                      title="Yearly Trend: Average View Growth (2020-2026)",
                      markers=True, template="plotly_dark")
    fig_year.show()

except Exception as e:
    print(f"Update: Still a column mismatch? Error: {e}")
    # This will help us see the REAL names if it fails again
    print(f"Country Columns: {df_country.columns.tolist()}")
    print(f"Category Columns: {df_category.columns.tolist()}")

In [12]:
import pandas as pd
import plotly.express as px

# 1. PATHS
base_path = "/kaggle/input/datasets/meruvakodandasuraj/youtube-trending-videos-20202026/"
video_path = base_path + "trending_videos.csv"
country_path = base_path + "country_summary.csv"

try:
    df = pd.read_csv(video_path)
    df_country = pd.read_csv(country_path)

    # 2. AUTO-DETECT COLUMNS (The Fail-Proof Part)
    # We check what the dataset actually calls 'likes' and 'comments'
    cols = df.columns.tolist()
    like_col = 'likes' if 'likes' in cols else ('total_likes' if 'total_likes' in cols else None)
    comm_col = 'comment_count' if 'comment_count' in cols else ('comments' if 'comments' in cols else None)
    view_col = 'views' if 'views' in cols else 'total_views'

    print(f"Detected columns: Likes={like_col}, Comments={comm_col}, Views={view_col}")

    # 3. CALCULATE ENGAGEMENT SAFELY
    if like_col and view_col:
        # If comments are missing, we just use likes
        comm_data = df[comm_col] if comm_col else 0
        df['engagement_rate'] = ((df[like_col] + comm_data) / df[view_col].replace(0, 1)) * 100
    else:
        df['engagement_rate'] = 0
        print("Warning: Could not find enough data for engagement calculation.")

    # 4. VISUALIZATION: Category Popularity
    # Using 'top_category' from your previous error message list
    fig_bar = px.bar(df_country, 
                     x='trending_country', 
                     y='avg_engagement',
                     color='top_category',
                     title="Average Engagement by Country and Top Category",
                     template="plotly_dark")
    fig_bar.show()

except Exception as e:
    print(f"Final Debug - Available Columns in Video DF: {df.columns.tolist()}")
    print(f"Error details: {e}")

Detected columns: Likes=likes, Comments=comments, Views=views
